## Install Dependencies

In [1]:
import sys
!{sys.executable} -m pip install -q --upgrade streamlit google-generativeai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 40.9 MB/s eta 0:00:00


In [12]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import io
import google.generativeai as genai
from collections import deque

st.set_page_config(layout="wide", page_title="Cognitive IIoT Dashboard", page_icon="🏭")

FAULT_THRESHOLD  = 8145.0
WARN_THRESHOLD   = FAULT_THRESHOLD * 0.88   # 88% → yellow warning zone
WINDOW_SIZE      = 80

# ── SIDEBAR ───────────────────────────────────────────────────────────────────
with st.sidebar:
    st.markdown("## ⚙️ System Initialization")
    api_key       = st.text_input("🔑 Gemini API Key:", type="password")
    uploaded_file = st.file_uploader("📂 Upload train_FD001.txt", type=["txt"])
    st.markdown("---")
    st.markdown("## 📊 Simulation Settings")
    op_load = st.slider("Operational Load (%)", 10, 100, 50, 1,
                        help="Higher load stresses the engine faster.")

if not api_key or not uploaded_file:
    st.title("🏭 Cognitive Digital Twin — NASA Turbofan Unit-01")
    st.info("👈 Enter your Gemini API Key and upload train_FD001.txt in the sidebar.")
    st.stop()

# ── GEMINI ────────────────────────────────────────────────────────────────────
@st.cache_resource
def get_model(key: str):
    genai.configure(api_key=key)
    # Try newest models first, fall back gracefully
    for name in ["gemini-2.0-flash", "gemini-1.5-flash-latest",
                 "gemini-1.5-flash", "gemini-1.0-pro"]:
        try:
            return genai.GenerativeModel(name), name
        except Exception:
            continue
    return None, "unavailable"

model, model_name = get_model(api_key)

# ── DATA ──────────────────────────────────────────────────────────────────────
@st.cache_data(show_spinner="Loading telemetry...")
def load_data(raw: bytes) -> pd.DataFrame:
    cols = ["unit", "cycle", "os1", "os2", "os3"]
    cols += [f"s{i}" for i in range(1, 22)]
    df = pd.read_csv(io.BytesIO(raw), sep=r"\s+", header=None, names=cols)
    return df[df["unit"] == 1].reset_index(drop=True)

df          = load_data(uploaded_file.read())
TOTAL_CYCLES = len(df)
# Baseline = median of first 10 cycles (stable running condition)
S14_BASELINE = float(df["s14"].iloc[:10].median())
S11_BASELINE = float(df["s11"].iloc[:10].median())

def compute_efficiency(s14: float) -> float:
    """Efficiency = how far we are from fault, expressed as 0–100%."""
    degradation = max(0.0, s14 - S14_BASELINE)
    max_degradation = FAULT_THRESHOLD - S14_BASELINE
    eff = 100.0 * (1.0 - degradation / max_degradation)
    return round(max(0.0, min(100.0, eff)), 1)

# ── SESSION STATE ─────────────────────────────────────────────────────────────
_DEFAULTS = {
    "running":          False,
    "fault_detected":   False,
    "anomaly_injected": False,
    "current_index":    0,
    "s11_buf":          [], # Changed to standard list to keep ALL historical data
    "s14_buf":          [], # Changed to standard list
    "eff_buf":          [], # Changed to standard list
    "cycle_buf":        [], # Changed to standard list
    "ai_report":        None,
    "peak_efficiency":  100.0,
    "fault_cycle":      None,
    "fan_angle":        0.0,
}
for k, v in _DEFAULTS.items():
    if k not in st.session_state:
        st.session_state[k] = v

# ── FAN SVG ───────────────────────────────────────────────────────────────────
def fan_svg(speed: float, state: str = "normal", angle: float = 0.0) -> str:
    # Safely building the SVG strings to prevent HTML bleed-over in the UI
    if state == "fault":
        colour, bg = "#F44336", "#2b1c1c"
        extra_circle = '<circle cx="50" cy="50" r="47" fill="none" stroke="#F44336" stroke-width="2"/>'
    elif state == "warn":
        colour, bg = "#FF9800", "#2b2000"
        extra_circle = '<circle cx="50" cy="50" r="47" fill="none" stroke="#FF9800" stroke-width="2"/>'
    else:
        colour, bg = "#00B4D8", "#1a1a2e"
        extra_circle = ''

    anim = f"animation: spin {speed:.2f}s infinite linear;" if speed > 0 else ""
    return f"""
    <div style="display:flex;flex-direction:column;justify-content:center;
                align-items:center;background:{bg};border-radius:12px;
                border:1px solid #333;height:280px;gap:6px;">
      <svg width="200" height="200" viewBox="0 0 100 100" xmlns="http://www.w3.org/2000/svg">
        <style>
          .blade {{ transform-origin:50px 50px; {anim} }}
          @keyframes spin {{ from{{transform:rotate({angle}deg)}} to{{transform:rotate({angle + 360}deg)}} }}
        </style>
        <circle cx="50" cy="50" r="48" fill="none" stroke="#555" stroke-width="3"/>
        <circle cx="50" cy="50" r="45" fill="#111"/>
        <g class="blade">
          <path d="M50 50 L50 5  A45 45 0 0 1 65 7  Z" fill="{colour}" opacity="0.9"/>
          <path d="M50 50 L50 95 A45 45 0 0 1 35 93 Z" fill="{colour}" opacity="0.9"/>
          <path d="M50 50 L95 50 A45 45 0 0 1 93 65 Z" fill="{colour}" opacity="0.9"/>
          <path d="M50 50 L5  50 A45 45 0 0 1 7  35 Z" fill="{colour}" opacity="0.9"/>
          <path d="M50 50 L81.8 18.2 A45 45 0 0 1 90 28  Z" fill="{colour}" opacity="0.85"/>
          <path d="M50 50 L18.2 81.8 A45 45 0 0 1 10 72  Z" fill="{colour}" opacity="0.85"/>
          <path d="M50 50 L18.2 18.2 A45 45 0 0 0 28 10  Z" fill="{colour}" opacity="0.85"/>
          <path d="M50 50 L81.8 81.8 A45 45 0 0 0 72 90  Z" fill="{colour}" opacity="0.85"/>
          <circle cx="50" cy="50" r="13" fill="#555"/>
          <circle cx="50" cy="50" r="5"  fill="#111"/>
        </g>
        {extra_circle}
      </svg>
    </div>"""

def efficiency_badge(eff: float) -> str:
    if eff >= 75:
        color, label = "#4CAF50", "HEALTHY"
    elif eff >= 50:
        color, label = "#FF9800", "DEGRADING"
    elif eff >= 25:
        color, label = "#FF5722", "CRITICAL"
    else:
        color, label = "#F44336", "FAILURE IMMINENT"
    return (f"<div style='text-align:center;padding:10px 0;'>"
            f"<div style='font-size:3rem;font-weight:bold;color:{color};'>{eff:.1f}%</div>"
            f"<div style='font-size:0.9rem;color:{color};letter-spacing:2px;'>{label}</div>"
            f"</div>")

# ── PAGE HEADER ───────────────────────────────────────────────────────────────
st.title("🏭 Cognitive Digital Twin — NASA Turbofan Unit-01")

# ── CONTROL PANEL ─────────────────────────────────────────────────────────────
st.markdown("### 🎛️ Edge Control Panel")
cp1, cp2, cp3 = st.columns(3)

with cp1:
    if st.button("▶️ Power ON System",
                 disabled=st.session_state.running or st.session_state.fault_detected,
                 use_container_width=True):
        st.session_state.running          = True
        st.session_state.fault_detected   = False
        st.session_state.anomaly_injected = False
        st.rerun()

with cp2:
    if st.button("⚠️ Inject Fault Anomaly",
                 disabled=not st.session_state.running or st.session_state.fault_detected,
                 use_container_width=True):
        st.session_state.anomaly_injected = True

with cp3:
    if st.button("🔄 Reset System", use_container_width=True):
        for k, v in _DEFAULTS.items():
            st.session_state[k] = v
        st.rerun()

st.markdown("---")

# ── LIVE DASHBOARD FRAGMENT ───────────────────────────────────────────────────
@st.fragment(run_every=0.3)
def live_dashboard():
    left, right = st.columns([1, 2])

    # ── OFFLINE OR COMPLETED STATE ─────────────────────────────────────────
    if not st.session_state.running and not st.session_state.fault_detected:
        # If we have data in the buffer, it means the simulation finished or paused.
        # We must preserve the last known state!
        if st.session_state.cycle_buf:
            last_s14 = list(st.session_state.s14_buf)[-1]
            last_eff = list(st.session_state.eff_buf)[-1]
            last_state = "warn" if last_s14 > WARN_THRESHOLD else "normal"

            with left:
                st.markdown(fan_svg(0, last_state, st.session_state.fan_angle), unsafe_allow_html=True)
                st.markdown(efficiency_badge(last_eff), unsafe_allow_html=True)
            with right:
                if st.session_state.current_index >= TOTAL_CYCLES:
                    st.success("✅ Simulation complete — Final engine state preserved for analysis.")
                else:
                    st.info("⚙️ System Paused.")
                _render_chart()
                _render_metrics()
        else:
            # True offline state (just reset or booted up)
            with left:
                st.markdown(fan_svg(0, "normal", st.session_state.fan_angle), unsafe_allow_html=True)
                st.markdown(efficiency_badge(100.0), unsafe_allow_html=True)
            with right:
                st.info("⚙️ System Offline — press **Power ON System** to begin simulation.")
        return

    # ── FAULT STATE ───────────────────────────────────────────────────────
    if st.session_state.fault_detected:
        eff = list(st.session_state.eff_buf)[-1] if st.session_state.eff_buf else 0.0
        with left:
            st.markdown(fan_svg(0, "fault", st.session_state.fan_angle), unsafe_allow_html=True)
            st.markdown(efficiency_badge(eff), unsafe_allow_html=True)
        with right:
            st.error(f"🚨 **CRITICAL FAULT DETECTED** at Cycle {st.session_state.fault_cycle} "
                     f"— LPC Core Speed exceeded {FAULT_THRESHOLD:.0f} RPM safety limit.")
            _render_chart()
            _render_metrics()
        if st.session_state.ai_report:
            _render_report()
        elif model:
            st.warning("⏳ Generating AI diagnostic report...")
        return

    # ── ADVANCE ONE TICK ──────────────────────────────────────────────────
    idx = st.session_state.current_index
    if idx >= TOTAL_CYCLES:
        st.session_state.running = False
        st.rerun() # Forces the UI to immediately jump into the "Completed" state above
        return

    row     = df.iloc[idx]

    # 🚀 MATH FIX: Additive stress instead of multiplication.
    # Every 1% above 50 adds 0.3 RPM of heat/stress to the engine.
    # At 100% load, it adds 15 extra RPM, pushing it dangerously close to failure without instantly exploding.
    stress_offset = (op_load - 50.0) * 0.3

    s14_val = (8250.0 if st.session_state.anomaly_injected
               else float(row["s14"]) + stress_offset)

    s11_val = float(row["s11"])
    cycle   = int(row["cycle"])
    eff     = compute_efficiency(s14_val)

    st.session_state.s11_buf.append(s11_val)
    st.session_state.s14_buf.append(s14_val)
    st.session_state.eff_buf.append(eff)
    st.session_state.cycle_buf.append(cycle)
    st.session_state.current_index = idx + 1

    # Track efficiency history for degradation rate
    if eff < st.session_state.peak_efficiency:
        st.session_state.peak_efficiency = eff

    # ── FAULT TRIGGER ─────────────────────────────────────────────────────
    if s14_val > FAULT_THRESHOLD:
        st.session_state.fault_detected   = True
        st.session_state.running          = False
        st.session_state.anomaly_injected = False
        st.session_state.fault_cycle      = cycle

        if model:
            eff_list  = list(st.session_state.eff_buf)
            recent_df = pd.DataFrame({
                "cycle":      list(st.session_state.cycle_buf)[-8:],
                "sensor_11":  list(st.session_state.s11_buf)[-8:],
                "sensor_14":  list(st.session_state.s14_buf)[-8:],
                "efficiency": list(st.session_state.eff_buf)[-8:],
            })
            # Degradation rate = efficiency drop per cycle over last 20 readings
            eff_window = list(st.session_state.eff_buf)[-20:]
            deg_rate   = round((eff_window[0] - eff_window[-1]) / max(len(eff_window), 1), 3)

            prompt = (
                f"You are an aerospace predictive maintenance engineer.\n\n"
                f"NASA CMAPSS Turbofan Engine — CRITICAL FAULT REPORT\n"
                f"{'='*50}\n"
                f"Fault detected at: Cycle {cycle}\n"
                f"Sensor 14 (LPC Core Speed): {s14_val:.1f} RPM (limit: {FAULT_THRESHOLD:.0f} RPM)\n"
                f"Final Operational Efficiency: {eff:.1f}%\n"
                f"Efficiency Degradation Rate: {deg_rate:.3f}% per cycle\n"
                f"Operational Load at fault: {op_load}%\n\n"
                f"Last 8 telemetry cycles before fault:\n"
                f"{recent_df.to_string(index=False)}\n\n"
                f"Provide a concise structured report with these exact sections:\n"
                f"## 🔍 Root Cause Analysis\n"
                f"## 🚨 Immediate Actions Required\n"
                f"## 🔧 Maintenance Recommendations\n"
                f"## 📈 Operational Efficiency Improvement Plan\n"
                f"Keep each section to 3-4 bullet points. Be specific and technical."
            )
            try:
                st.session_state.ai_report = model.generate_content(prompt).text
            except Exception as exc:
                st.session_state.ai_report = f"⚠️ AI report unavailable: {exc}"
        return

    # ── RENDER NOMINAL STATE ──────────────────────────────────────────────
    state = "warn" if s14_val > WARN_THRESHOLD else "normal"
    spd   = max(0.08, 1.0 - (op_load / 110.0))

    # Advance the fan angle mathematically based on the exact 0.3s update interval
    st.session_state.fan_angle = (st.session_state.fan_angle + (360 / spd) * 0.3) % 360

    with left:
        st.markdown(fan_svg(spd, state, st.session_state.fan_angle), unsafe_allow_html=True)
        st.markdown(efficiency_badge(eff), unsafe_allow_html=True)

    with right:
        if state == "warn":
            st.warning(f"⚠️ **DEGRADATION WARNING** — Cycle {cycle} | "
                       f"Sensor 14 at {s14_val:.0f} RPM ({(s14_val/FAULT_THRESHOLD*100):.1f}% of fault limit)")
        else:
            st.success(f"✅ **NOMINAL** | Cycle: {cycle} | Load: {op_load}% | "
                       f"{'🔴 ANOMALY ARMED' if st.session_state.anomaly_injected else '🟢 All Systems Normal'}")
        _render_chart()
        _render_metrics()

def _render_chart():
    if not st.session_state.cycle_buf:
        return
    idx = list(st.session_state.cycle_buf)

    # Create two tabs to separate the data scales
    tab1, tab2 = st.tabs(["📈 LPC Speed (Sensor 14)", "📊 Efficiency & Fan Speed"])

    with tab1:
        # 🚀 MAGIC FIX: Use Altair to force the Y-Axis to NOT start at 0
        import altair as alt

        s14_df = pd.DataFrame({
            "Cycle": idx,
            "LPC Speed (RPM)": list(st.session_state.s14_buf)
        })

        # Build a custom chart that strictly zooms into the active data range
        chart = alt.Chart(s14_df).mark_line(color="#F44336").encode(
            x=alt.X("Cycle", title="Engine Cycle"),
            y=alt.Y("LPC Speed (RPM)", scale=alt.Scale(zero=False)) # <--- The command that fixes the flat line
        ).properties(height=350)

        st.altair_chart(chart, use_container_width=True)

    with tab2:
        chart_df = pd.DataFrame({
            "Sensor 11 — Fan Speed": list(st.session_state.s11_buf),
            "Efficiency (%)": list(st.session_state.eff_buf),
        }, index=idx)
        st.line_chart(chart_df, height=350)

def _render_metrics():
    if not st.session_state.s14_buf:
        return
    s14    = list(st.session_state.s14_buf)[-1]
    s11    = list(st.session_state.s11_buf)[-1]
    eff    = list(st.session_state.eff_buf)[-1]
    cycle  = list(st.session_state.cycle_buf)[-1]

    # Degradation rate (efficiency % lost per cycle, last 10 readings)
    eff_w  = list(st.session_state.eff_buf)
    deg    = round((eff_w[0] - eff_w[-1]) / max(len(eff_w), 1), 3) if len(eff_w) > 1 else 0.0

    # Estimated cycles to fault (linear projection)
    margin = FAULT_THRESHOLD - s14
    if deg > 0 and len(eff_w) > 5:
        # approximate: how many more % efficiency to lose before fault?
        pct_to_fault = eff
        cycles_left  = int(pct_to_fault / deg) if deg > 0 else 9999
    else:
        cycles_left = 9999

    m1, m2, m3, m4, m5 = st.columns(5)
    m1.metric("🔄 Cycle",           cycle)
    m2.metric("🌀 Fan Speed S11",   f"{s11:.1f}")
    m3.metric("⚡ LPC Speed S14",   f"{s14:.0f}",
              delta="⚠️ WARNING" if s14 > WARN_THRESHOLD else "Normal",
              delta_color="inverse")
    m4.metric("📉 Degradation Rate", f"{deg:.3f}%/cyc")
    m5.metric("⏳ Est. Life Left",
              f"{cycles_left} cyc" if cycles_left < 9999 else "Stable")

def _render_report():
    report = st.session_state.ai_report
    if not report:
        return
    st.markdown("---")
    st.markdown("### 🤖 AI Diagnostic Report")
    st.markdown(
        f"<div style='background:#1a1a1a;padding:24px;border-radius:12px;"
        f"border-left:5px solid #F44336;color:#eee;line-height:1.8;"
        f"font-size:0.95rem;'>{report.replace(chr(10), '<br>')}</div>",
        unsafe_allow_html=True)

live_dashboard()

Overwriting app.py


In [14]:
import sys, time, urllib.request

!pkill -f streamlit 2>/dev/null
!pkill -f cloudflared 2>/dev/null
time.sleep(2)

# Use the exact same Python that has streamlit installed
!nohup {sys.executable} -m streamlit run app.py \
    --server.port=8501 \
    --server.enableCORS=false \
    --server.enableXsrfProtection=false \
    --server.headless=true \
    > /content/streamlit.log 2>&1 &

print("⏳ Waiting for Streamlit...")
for i in range(30):
    try:
        urllib.request.urlopen("http://localhost:8501/_stcore/health", timeout=1)
        print("✅ Streamlit is UP")
        break
    except Exception:
        time.sleep(1)
else:
    print("❌ Failed. Logs:")
    !tail -20 /content/streamlit.log

!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64
print("="*55)
print("🚀 Tunnel starting — look for .trycloudflare.com URL")
print("="*55)
!./cloudflared-linux-amd64 tunnel --url http://localhost:8501

^C
^C
⏳ Waiting for Streamlit...
✅ Streamlit is UP
🚀 Tunnel starting — look for .trycloudflare.com URL
2026-04-14T21:47:09Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-04-14T21:47:09Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-04-14T21:47:13Z INF +--------------------------------------------------------------------------------------------+
2026-04-14T21:47:13Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time